# 2D Resonance Imaging with PLEIADES

This notebook demonstrates the complete 2D imaging pipeline for
spatially-resolved neutron resonance analysis. Each pixel of a
hyperspectral neutron transmission image is fitted independently using
SAMMY, and the results are aggregated into isotope abundance maps.

## Pipeline Overview

```
TIFF data  →  HyperspectralLoader  →  pixel spectra
                                           ↓
                              BatchFittingOrchestrator (parallel SAMMY)
                                           ↓
                                   ResultsAggregator  →  2D maps
                                           ↓
                          AbundanceMapGenerator + AbundanceMapVisualizer
```

**Two usage levels:**
1. **High-level**: `analyze_imaging()` — single function call for the entire pipeline
2. **Component-level**: Step-by-step control over each stage

### Requirements

- **SAMMY executable** installed and on PATH (or provide full path)
- Test TIFF stack at `tests/data/pleiades_data/LANL-ORNL_example.tif`

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from pleiades.imaging import (
    HyperspectralLoader,
    ResultsAggregator,
    AbundanceMapGenerator,
    AbundanceMapVisualizer,
    Imaging2DResults,
    analyze_imaging,
)
from pleiades.imaging.config import ImagingConfig
from pleiades.imaging.orchestrator import BatchFittingOrchestrator

## 1. Locate SAMMY Executable

The 2D imaging pipeline requires a working SAMMY installation. On ORNL systems
it is typically at `/SNS/software/sammy/bin/sammy`.

In [ ]:
# Try to find SAMMY on PATH
sammy_path_str = "/Users/8cz/code-int.ornl.gov/sammy/build/bin/sammy"

sammy_executable = Path(sammy_path_str)
print(f"SAMMY executable: {sammy_executable}")

## 2. Load Hyperspectral Data

The `HyperspectralLoader` reads multi-page TIFF files where each page is one
energy bin and the 2D spatial dimensions form the image.

In [ ]:
tiff_path = Path("../../tests/data/pleiades_data/LANL-ORNL_example.tif")

# Energy axis for the LANL-ORNL test data.
# Reverse-engineered by matching observed resonance dip positions (bin indices)
# to known Ta-181 ENDF resonance energies. Linear fit through 16 matched pairs
# gives E(eV) = 0.6613 * bin + 13.005 with R^2 = 0.99998.
energy = np.linspace(13.0, 343.0, 500)

loader = HyperspectralLoader(tiff_path, energy=energy)
hyperspectral = loader.load()

n_energy, height, width = hyperspectral.shape
print(f"Image shape: {height}x{width} pixels, {n_energy} energy bins")
print(f"Total pixels: {hyperspectral.n_pixels:,}")
print(f"Energy range: {energy[0]:.1f} - {energy[-1]:.1f} eV")

### Visualize the raw data

Look at the spatial image at a single energy bin, and a single pixel’s
transmission spectrum.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Spatial image at a selected energy bin
bin_idx = 140
im = axes[0].imshow(hyperspectral.data[bin_idx], cmap="viridis", origin="upper")
axes[0].set_title(f"Transmission at energy bin {bin_idx} ({energy[bin_idx]:.1f} eV)")
fig.colorbar(im, ax=axes[0], label="Transmission")

# Spectrum for center pixel
center_row, center_col = height // 2, width // 2
spectrum = hyperspectral.data[:, center_row, center_col]
axes[1].plot(energy, spectrum, linewidth=0.5)
axes[1].set_xlabel("Energy (eV)")
axes[1].set_ylabel("Transmission")
axes[1].set_title(f"Pixel ({center_row}, {center_col}) spectrum")

fig.tight_layout()
plt.show()

## 3. Configure Material and Isotopes

`ImagingConfig` holds the isotope list, material properties, and energy range
for the SAMMY fitting engine. This is the same configuration used for every
pixel in the image.

In [ ]:
config = ImagingConfig(
    isotopes=["Ta-181"],
    element="Ta",
    mass_number=181,
    density_g_cm3=16.65,
    thickness_mm=0.025,  # ~25 um; thin enough to avoid saturated resonances
    atomic_mass_amu=180.94788,
    natural_abundances=True,
    min_energy_eV=13.0,
    max_energy_eV=343.0,
    temperature_K=293.6,
)

print(f"Isotopes: {config.isotopes}")
print(f"Material: {config.element}-{config.mass_number}")
print(f"  Density: {config.density_g_cm3} g/cm\u00b3")
print(f"  Thickness: {config.thickness_mm} mm")
print(f"Energy range: {config.min_energy_eV}-{config.max_energy_eV} eV")
print(f"Abundances: {config.get_abundances()}")

## 4. Batch Pixel Fitting with SAMMY

The `BatchFittingOrchestrator` is the core of the 2D imaging pipeline. For
each pixel it:

1. Exports the pixel spectrum to CSV
2. Converts CSV → SAMMY `.twenty` format
3. Creates a SAMMY input file (`.inp`) from the `ImagingConfig`
4. Stages the JSON config + ENDF resonance parameter files
5. Executes SAMMY via `LocalSammyRunner`
6. Parses the `SAMMY.LPT` output for fitted abundances and χ²

All of this runs in parallel across `n_workers` processes.

We demonstrate on a small 4×4 ROI (16 pixels) to keep run time short.

In [ ]:
# Extract a small ROI for demonstration
roi = (120, 120, 124, 124)  # (x1, y1, x2, y2) → 4×4 = 16 pixels
pixels = list(loader.iter_pixels(roi=roi))

print(f"ROI pixels: {len(pixels)}")
print(f"Pixel grid: rows {pixels[0].row}–{pixels[-1].row}, cols {pixels[0].col}–{pixels[-1].col}")
print(f"Energy points per pixel: {len(pixels[0].energy)}")

In [ ]:
# Create the orchestrator
orchestrator = BatchFittingOrchestrator(
    imaging_config=config,
    sammy_executable=sammy_executable,
    n_workers=4,  # number of parallel SAMMY processes
)

# Fit all pixels in the ROI
# Each pixel gets its own SAMMY execution with isolated temp files
pixel_results = orchestrator.fit_pixels(
    pixels,
    timeout_per_job=60.0,  # seconds per pixel
    max_retries=1,         # retry failed pixels once
)

# Summarize results
n_success = sum(1 for r in pixel_results if r.success)
n_failed = len(pixel_results) - n_success
print(f"\nResults: {n_success} succeeded, {n_failed} failed out of {len(pixel_results)} pixels")

### Inspect individual pixel results

Each `PixelFitResult` contains the fitted nuclear parameters (isotope
abundances), chi-squared goodness of fit, and success/failure status.

In [ ]:
for result in pixel_results[:4]:  # show first 4
    status = "✓" if result.success else "✗"
    chi_str = f"{result.chi_squared:.2f}" if result.chi_squared else "N/A"
    print(f"  Pixel ({result.row}, {result.col}): {status}  χ²={chi_str}", end="")
    if result.success and result.fit_results:
        abundances = result.get_abundances()
        print(f"  abundances={[f'{a:.4f}' for a in abundances]}")
    elif result.error_message:
        print(f"  error: {result.error_message[:80]}")
    else:
        print()

## 5. Aggregate Results into 2D Maps

The `ResultsAggregator` takes the list of per-pixel fit results and
builds 2D abundance maps, a chi-squared map, and a boolean success mask.

Since we only fitted a small ROI, we aggregate over the full image dimensions
(unfitted pixels will be NaN).

In [ ]:
aggregator = ResultsAggregator(
    isotope_names=config.isotopes,
    height=height,
    width=width,
)
results = aggregator.aggregate(pixel_results, hyperspectral)

print(f"Abundance maps shape: {results.abundance_maps.shape}  (n_isotopes, height, width)")
print(f"Chi-squared map shape: {results.chi_squared_map.shape}")
print(f"Fitted pixels: {results.success_mask.sum()} / {results.success_mask.size}")
print(f"Isotopes: {results.isotope_names}")

## 6. Visualize Abundance Maps

`AbundanceMapGenerator` extracts individual isotope maps.
`AbundanceMapVisualizer` provides publication-quality plots.

In [ ]:
gen = AbundanceMapGenerator(results)
ta_map = gen.generate_map("Ta-181")

# Show just the ROI region where we have real data
roi_slice = ta_map[120:124, 120:124]
print(f"Ta-181 abundance in ROI:")
print(np.array2string(roi_slice, precision=4, suppress_small=True))

# Quality map
chi2_map = gen.generate_quality_map("chi_squared")
roi_chi2 = chi2_map[120:124, 120:124]
print(f"\nChi-squared in ROI:")
print(np.array2string(roi_chi2, precision=2, suppress_small=True))

In [ ]:
viz = AbundanceMapVisualizer(results)

# Plot the full image (most will be NaN; the ROI will have color)
fig, ax = viz.plot_single_isotope("Ta-181", cmap="plasma")
ax.set_title("Ta-181 Abundance Map (4×4 ROI fitted)")

# Highlight the ROI with a rectangle
from matplotlib.patches import Rectangle
rect = Rectangle((119.5, 119.5), 4, 4, linewidth=2, edgecolor="red", facecolor="none")
ax.add_patch(rect)
plt.show()

In [ ]:
# Quality overlay: chi-squared on top of abundance
fig, ax = viz.plot_quality_overlay("Ta-181", quality_metric="chi_squared", alpha=0.4)
ax.set_title("Ta-181 with χ² overlay")
plt.show()

## 7. Save and Reload Results

Results persist to HDF5 for later visualization or further analysis.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    h5_path = Path(tmpdir) / "imaging_results.h5"
    results.save_hdf5(h5_path)
    print(f"Saved to {h5_path.name} ({h5_path.stat().st_size / 1024:.0f} KB)")

    # Reload
    loaded = Imaging2DResults.load_hdf5(h5_path, source_hyperspectral=hyperspectral)
    print(f"Loaded isotopes: {loaded.isotope_names}")
    print(f"Maps shape: {loaded.abundance_maps.shape}")

    # Verify round-trip fidelity
    np.testing.assert_array_equal(results.abundance_maps, loaded.abundance_maps)
    print("Round-trip verification: PASSED")

## 8. High-Level API: `analyze_imaging()`

The entire pipeline above (load → iterate → fit → aggregate) collapses into
a single function call. This is the recommended entry point for production use.

In [ ]:
results_api = analyze_imaging(
    source=tiff_path,
    imaging_config=config,
    sammy_executable=sammy_executable,
    energy=energy,
    n_workers=4,
    roi=(126, 126, 130, 130),  # different 4×4 ROI
    timeout_per_job=60.0,
)

print(f"Success rate: {results_api.success_mask.sum()} / {results_api.success_mask.size} pixels")
print(f"Isotopes: {results_api.isotope_names}")
print(f"Abundance maps shape: {results_api.abundance_maps.shape}")

In [ ]:
# Visualize the high-level API results
viz_api = AbundanceMapVisualizer(results_api)
fig, ax = viz_api.plot_single_isotope("Ta-181", cmap="plasma")
ax.set_title("analyze_imaging() result")
plt.show()

### `analyze_imaging()` parameter reference

| Parameter | Description | Default |
|-----------|-------------|---------|
| `source` | Path to TIFF file or directory | *required* |
| `imaging_config` | Isotopes and material properties | *required* |
| `sammy_executable` | Path to SAMMY binary | *required* |
| `energy` | Energy axis (eV); inferred if `None` | `None` |
| `n_workers` | Parallel SAMMY processes | `4` |
| `roi` | Sub-region `(x1, y1, x2, y2)` | `None` (all) |
| `checkpoint_file` | Save/resume checkpoint | `None` |
| `resume` | Resume from existing checkpoint | `False` |
| `timeout_per_job` | Max seconds per pixel | `None` |
| `max_retries` | Retry failed pixels | `0` |
| `temp_manager` | Custom `TempFileManager` | auto-created |
| `save_path` | Auto-save results to HDF5 | `None` |

## Summary

This notebook demonstrated the full 2D resonance imaging pipeline:

1. **Load** hyperspectral TIFF data with `HyperspectralLoader`
2. **Configure** isotopes and material properties with `ImagingConfig`
3. **Fit** each pixel with SAMMY via `BatchFittingOrchestrator`
4. **Aggregate** per-pixel results into 2D maps with `ResultsAggregator`
5. **Visualize** abundance and quality maps with `AbundanceMapVisualizer`
6. **Save/load** results in HDF5 format
7. **One-call API** via `analyze_imaging()` for production use

For larger images, increase `n_workers` and use `checkpoint_file` +
`resume=True` to recover from interruptions.